# Task 5 - Credit Risk Model Training and Tracking

## Objectives

- Train multiple classification models
- Perform hyperparameter tuning
- Evaluate model performance
- Track experiments with MLflow
- Select and register the best model

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

import mlflow
import mlflow.sklearn

c:\Users\YENGU\credit-risk-model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load Data
df = pd.read_csv(
    "../data/processed/customer_with_target.csv"
)

df.head()

,CustomerId,total_transaction_amount,avg_transaction_amount,transaction_count,is_high_risk
0,CustomerId_1,-10000.0,-10000.000000,1,1
1,CustomerId_10,-10000.0,-10000.000000,1,1
2,CustomerId_1001,20000.0,4000.000000,5,1
3,CustomerId_1002,4225.0,384.090909,11,0
4,CustomerId_1003,20000.0,3333.333333,6,0


In [3]:
# Prepare Features
X = df.drop(
    columns=[
        "CustomerId",
        "is_high_risk"
    ]
)

X = pd.get_dummies(
    X,
    drop_first=True
)

y = df["is_high_risk"]

print(X.shape)
print(y.shape)

(3742, 3)
(3742,)


In [4]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(2993, 3)
(749, 3)


In [6]:
# Helper Function
def evaluate_model(model):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:,1]

    metrics = {
        "accuracy":
            accuracy_score(y_test, y_pred),

        "precision":
            precision_score(y_test, y_pred),

        "recall":
            recall_score(y_test, y_pred),

        "f1":
            f1_score(y_test, y_pred),

        "roc_auc":
            roc_auc_score(y_test, y_prob)
    }

    return metrics

In [7]:
# Logistic Regression
logistic = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic.fit(
    X_train,
    y_train
)

log_metrics = evaluate_model(
    logistic
)

log_metrics

{'accuracy': 0.6181575433911882,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.752514367816092}

In [8]:
# Random Forest
rf = RandomForestClassifier(
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

rf_metrics = evaluate_model(
    rf
)

rf_metrics

{'accuracy': 0.7049399198931909,
 'precision': 0.6019108280254777,
 'recall': 0.6631578947368421,
 'f1': 0.6310517529215359,
 'roc_auc': 0.7534671808832427}

In [9]:
# Hyperparameter Tuning
param_grid = {
    "n_estimators":[100,200],
    "max_depth":[5,10,15]
}

grid_search = GridSearchCV(
    RandomForestClassifier(
        random_state=42
    ),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

grid_search.fit(
    X_train,
    y_train
)

best_rf = grid_search.best_estimator_

grid_search.best_params_

{'max_depth': 10, 'n_estimators': 100}

In [10]:
# Evaluate Tuned Model
best_rf_metrics = evaluate_model(
    best_rf
)

best_rf_metrics

{'accuracy': 0.7222963951935915,
 'precision': 0.6230031948881789,
 'recall': 0.6842105263157895,
 'f1': 0.6521739130434783,
 'roc_auc': 0.7826262855414399}

In [11]:
# Compare Results
results = pd.DataFrame([
    {
        "Model":"Logistic Regression",
        **log_metrics
    },
    {
        "Model":"Random Forest",
        **rf_metrics
    },
    {
        "Model":"Tuned Random Forest",
        **best_rf_metrics
    }
])

results

,Model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.618158,0.000000,0.000000,0.000000,0.752514
1,Random Forest,0.704940,0.601911,0.663158,0.631052,0.753467
2,Tuned Random Forest,0.722296,0.623003,0.684211,0.652174,0.782626


In [12]:
# MLflow Tracking Example
mlflow.set_experiment(
    "Credit Risk Modeling"
)

2026/06/04 11:29:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/04 11:29:19 INFO mlflow.store.db.utils: Updating database tables
2026/06/04 11:30:19 INFO mlflow.tracking.fluent: Experiment with name 'Credit Risk Modeling' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/YENGU/credit-risk-model/notebooks/mlruns/1', creation_time=1780561819630, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780561819630, lifecycle_stage='active', name='Credit Risk Modeling', tags={}, trace_location=None, workspace='default'>

In [13]:
with mlflow.start_run(
    run_name="TunedRandomForest"
):

    mlflow.log_params(
        grid_search.best_params_
    )

    mlflow.log_metrics(
        best_rf_metrics
    )

    mlflow.sklearn.log_model(
        best_rf,
        artifact_path="model",
        registered_model_name="credit_risk_model"
    )

2026/06/04 11:33:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 11:33:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'credit_risk_model'.
Created version '1' of model 'credit_risk_model'.
